# Manuscript figures

The four figures of the methods paper. All four follow one nucleus through the pipeline:

| Figure | Shows | Drawn from |
|---|---|---|
| 1. Data and extracted parameters | the localizations of both marks, the nucleus mask and radial contours, and the statistics the simulation is built from (radial profile, within-domain RDF / ADF, pair correlation g(r)) | the nucleus, recomputed here with `hmt.measure` |
| 2. Building the simulation | domain centres → localizations drawn from the RDF / ADF → background, and the simulated nucleus checked against the real one | the drivers' simulated nucleus (`calibration/<key>/<mark>_simulated.parquet`) |
| 3. Calibration | the simulated crop thinned to lower densities, the size-error search over (eps, min_samples), the fitted eps(density) and min_samples(density), and measured size vs density with fixed vs corrected parameters | the drivers' calibration (`<mark>_calibration.json`, `<mark>_size_vs_density.csv`), plus the size-error maps recomputed here |
| 4. Clustering the nucleus | the parameters read off the calibration at this nucleus's density, the clustered nucleus, and the same fixed-vs-corrected check on the real nucleus thinned to lower densities | the nucleus and its calibration, clustered here with `hmt.cluster` |

The nucleus is the one the drivers calibrated on (`CALIBRATION_KEY`), so the simulation in Fig. 2 and the
calibration in Fig. 3 are the ones built from the data in Fig. 1.

**The heavy steps come from the drivers** (`drivers/`, or `full_pipeline.ipynb`): the simulation and the
calibration. Re-run them to change those, and the figures update. The first code cell after the settings
checks whether the run is older than the current `hmt_v3` code and whether the calibration did its job.
Read its warnings before using the figures.

Each figure is saved as PDF (vector text, rasterized point clouds) and PNG (300 dpi) in
`RUN_DIR / "manuscript_figures"`, sized for a double-column page (`FIG_WIDTH_IN`). Run time is about
2 minutes.

**Plot conventions.**
- Colour always means the mark: H3K27me3 blue, H3K27ac orange (the first two slots of the validated
  categorical palette).
- Fixed DBSCAN parameters are dashed with open markers; density-corrected ones are solid with filled markers.
- Grey is background, unclustered localizations, or a random (CSR) reference.

In [ ]:
from pathlib import Path
import json
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from scipy.ndimage import gaussian_filter
from scipy.spatial import ConvexHull
from scipy.stats import linregress

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "hmt_v3").is_dir())
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "drivers"))
import hmt_v3 as hmt
import config as C  # the drivers' settings: marks, calibration and clustering parameters

## Settings

In [ ]:
# --- Which run to plot ----------------------------------------------------------
RUN_DIR = Path(C.OUTPUT_DIR)              # the drivers' output folder (analysis_output by default)
FIG_DIR = RUN_DIR / "manuscript_figures"
CALIBRATION_KEY = C.CONDITION_ORDER[0]    # the drivers' calibration to show; the figures follow its source nucleus
CHANNELS = list(C.CHANNELS)               # ("me3", "ac")
MARK_LABELS = dict(C.CHANNEL_LABELS)      # {"me3": "H3K27me3", "ac": "H3K27ac"}
WINDOW_NM = 2000.0                        # side of the zoom windows (Figs 1b, 2a-c, 4d-e)

# --- Statistics recomputed here (on random subsets; each is density-independent or rescaled) ---
N_PARAMS = 60_000                   # localizations per mark for the RDF / ADF
N_PCF = 30_000                      # localizations per mark for g(r)
RDF = dict(sdis=500, step=10)       # same as generate_nucleus
PCF = dict(r_max=1000.0, dr=20.0)
LANDSCAPE_MARK = "me3"              # Fig. 3e-f: the mark whose size-error search is shown
LANDSCAPE_FRACTIONS = (1.0, 0.1)    # ... at these density fractions
LANDSCAPE_EPS_NM = np.arange(10, 225, 15.0)  # the calibration's own coarse eps grid
CROP_FRACTIONS = (1.0, 0.5, 0.1)    # Fig. 3b-d: densities shown
REAL_FRACTIONS = np.round(np.linspace(1.0, 0.1, 10), 2)  # Fig. 4f-h: the real nucleus thinned to these
MIN_DOMAIN_LOCS = 10                # nanodomains smaller than this are left out of sizes and counts (as in the drivers)
RANDOM_SEED = 0

# --- Style ------------------------------------------------------------------------
FIG_WIDTH_IN = 7.2                  # double-column width
MARK_COLORS = {"me3": "#77DD76", "ac": "#FF6962"}   # categorical slots 1-2 (blue, orange)
METHOD_STYLE = {"fixed": dict(ls="--", filled=False), "corrected": dict(ls="-", filled=True)}
INK, INK_2, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
BACKGROUND_COLOR = "#b5b4ad"        # background / unclustered localizations
SEQ_RAMP = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]  # blue 100-700, odd steps

FIG_DIR.mkdir(parents=True, exist_ok=True)

### Plot helpers

The same chart style as `edu_s_phase_ripley.ipynb`: left-aligned titles, recessive grey grid and axes, and
text in ink rather than in the series colour. Font sizes are for a figure printed at `FIG_WIDTH_IN`.

In [ ]:
plt.rcParams.update({
    "font.size": 7, "axes.titlesize": 7.5, "axes.labelsize": 7, "xtick.labelsize": 6.5, "ytick.labelsize": 6.5,
    "legend.fontsize": 6.5, "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.major.size": 2.5, "ytick.major.size": 2.5, "lines.linewidth": 1.3, "lines.markersize": 4,
    "figure.dpi": 150, "savefig.dpi": 300, "pdf.fonttype": 42, "svg.fonttype": "none",
})
SEQ_CMAP = mcolors.LinearSegmentedColormap.from_list("seq_blue", SEQ_RAMP)


def style(ax, xlabel, ylabel, title):
    ax.set_xlabel(xlabel, color=INK_2)
    ax.set_ylabel(ylabel, color=INK_2)
    ax.set_title(title, loc="left", color=INK)
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color(AXIS)
    ax.tick_params(colors=MUTED, labelcolor=INK_2)
    ax.grid(True, color=GRID, lw=0.5)
    ax.set_axisbelow(True)


def image_panel(ax, title, scalebar_um, bar_color="white"):
    # Image-like panel: no axes, a scale bar (data units are um) in the lower right
    ax.set_title(title, loc="left", color=INK)
    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    xs = x1 - 0.06 * (x1 - x0) - scalebar_um
    ys = y0 + 0.06 * (y1 - y0)
    ax.plot([xs, xs + scalebar_um], [ys, ys], color=bar_color, lw=2, solid_capstyle="butt")
    ax.text(xs + scalebar_um / 2, ys + 0.025 * (y1 - y0), f"{scalebar_um:g} µm", color=bar_color,
            ha="center", va="bottom", fontsize=6)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)


def letter_panels(axes):
    # Bold panel letter at the start of each panel's left-aligned title
    for letter, ax in zip("abcdefghijklmnop", axes):
        ax.set_title(rf"$\bf{{{letter}}}$   " + ax.get_title(loc="left"), loc="left", color=INK)


def save(fig, name):
    fig.savefig(FIG_DIR / f"{name}.pdf")
    fig.savefig(FIG_DIR / f"{name}.png", dpi=300)


def um(nm):
    return nm / 1e3


def render(locs_by_mark, extent_nm, px_nm, blur_px=1.0, gamma=0.7):
    # Two-colour localization image: each mark is histogrammed, blurred, scaled to its 99.5th percentile and
    # added in its colour on black (overlap tends to white). Rows are y, so imshow with origin="lower".
    x0, x1, y0, y1 = extent_nm
    x_edges, y_edges = np.arange(x0, x1 + px_nm, px_nm), np.arange(y0, y1 + px_nm, px_nm)
    img = np.zeros((len(y_edges) - 1, len(x_edges) - 1, 3))
    for ch, df in locs_by_mark.items():
        h = gaussian_filter(np.histogram2d(df["y [nm]"], df["x [nm]"], bins=(y_edges, x_edges))[0], blur_px)
        scale = np.percentile(h[h > 0], 99.5) if np.any(h > 0) else 1.0
        img += (np.clip(h / scale, 0, 1) ** gamma)[..., None] * np.array(mcolors.to_rgb(MARK_COLORS[ch]))
    return np.clip(img, 0, 1)


def show_render(ax, locs_by_mark, extent_nm, px_nm, legend=False, **kw):
    x0, x1, y0, y1 = extent_nm
    ax.set_facecolor("black")
    ax.imshow(render(locs_by_mark, extent_nm, px_nm, **kw), origin="lower",
              extent=(um(x0), um(x1), um(y0), um(y1)), interpolation="nearest")
    if legend:
        handles = [Line2D([], [], ls="", marker="s", color=MARK_COLORS[ch], label=MARK_LABELS[ch]) for ch in locs_by_mark]
        ax.legend(handles=handles, loc="upper left", frameon=False, labelcolor="white", handletextpad=0.2,
                  borderaxespad=0.3, fontsize=6)


def scatter_locs(ax, df, color, s=0.5, **kw):
    # Localizations as points, rasterized so a PDF stays small
    ax.scatter(um(df["x [nm]"]), um(df["y [nm]"]), s=s, color=color, linewidths=0, rasterized=True, **kw)


def window(centre, size_nm=WINDOW_NM):
    cx, cy = centre
    return (cx - size_nm / 2, cx + size_nm / 2, cy - size_nm / 2, cy + size_nm / 2)


def in_window(df, extent_nm):
    x0, x1, y0, y1 = extent_nm
    return df[df["x [nm]"].between(x0, x1) & df["y [nm]"].between(y0, y1)]


def set_window(ax, extent_nm):
    ax.set_xlim(um(extent_nm[0]), um(extent_nm[1]))
    ax.set_ylim(um(extent_nm[2]), um(extent_nm[3]))
    ax.set_aspect("equal")


def nucleus_extent(cell, pad_nm=500.0):
    # Bounding box of the mask, padded, in nm
    xi, yi = np.nonzero(cell["mask"])
    b = cell["bin_nm"]
    return (cell["x_min"] + xi.min() * b - pad_nm, cell["x_min"] + (xi.max() + 1) * b + pad_nm,
            cell["y_min"] + yi.min() * b - pad_nm, cell["y_min"] + (yi.max() + 1) * b + pad_nm)


def mask_outline(ax, cell, color="white"):
    b = cell["bin_nm"]
    xc = um(cell["x_min"] + (np.arange(cell["mask"].shape[0]) + 0.5) * b)
    yc = um(cell["y_min"] + (np.arange(cell["mask"].shape[1]) + 0.5) * b)
    ax.contour(xc, yc, cell["mask"].T.astype(float), levels=[0.5], colors=color, linewidths=0.6)

## Load the run

Loads the nucleus (masked localizations, mask, contour bands), its calibration and the simulated nucleus.
The drivers must have been run with `CALIBRATION_MODE = "per_condition"` (their default), so the
calibration lives in `calibration/<CALIBRATION_KEY>/`.

**Check before using the figures.** The table summarises the calibration of each mark. A calibration is
doing its job when:
- most density fractions are fitted;
- the ground-truth domains are mostly separate (`gt_overlap` low);
- the measured size depends less on density with corrected parameters than with fixed ones
  (`r2_corrected` < `r2_fixed`). These R² are the drivers', on bounding-box size; Fig. 3i–j shows the same
  check on the p75 diameter, so its R² differ a little.

The warnings name anything that fails, and any `hmt_v3` file changed after the run.

In [ ]:
cal_dir = RUN_DIR / "calibration" / CALIBRATION_KEY
if not cal_dir.is_dir():
    raise FileNotFoundError(f"{cal_dir} not found; run the drivers with CALIBRATION_MODE = 'per_condition'")
calib = {ch: json.load(open(cal_dir / f"{ch}_calibration.json")) for ch in CHANNELS}
size_tables = {ch: pd.read_csv(cal_dir / f"{ch}_size_vs_density.csv") for ch in CHANNELS
               if (cal_dir / f"{ch}_size_vs_density.csv").exists()}
sims = {ch: pd.read_parquet(cal_dir / f"{ch}_simulated.parquet") for ch in CHANNELS}
info = {ch: calib[ch]["generate_nucleus"] for ch in CHANNELS}  # what the drivers' simulation measured and used

cell_id = calib[CHANNELS[0]]["source_cell"]
geom = np.load(RUN_DIR / "cells" / cell_id / "mask.npz")
cell = {ch: pd.read_parquet(RUN_DIR / "cells" / cell_id / f"{ch}_masked.parquet") for ch in CHANNELS}
cell.update(mask=geom["binary_mask"], bands=geom["contour_bands"], x_min=float(geom["x_min"]),
            y_min=float(geom["y_min"]), bin_nm=float(geom["bin_nm"]), area_nm2=float(geom["area_nm2"]))

# --- run checks ---
qc_rows, warnings = [], []
for ch in CHANNELS:
    c = calib[ch]
    r2 = c.get("size_vs_density_r2", {})
    qc_rows.append({"mark": MARK_LABELS[ch], "fractions_fit": f"{len(c['densities'])}/{len(c['density_fractions'])}",
                    "gt_diameter_nm": c["gt_target"], "gt_overlap": c.get("gt_frac_domains_overlapping", np.nan),
                    "r2_fixed": r2.get("fixed", np.nan), "r2_corrected": r2.get("corrected", np.nan)})
    if not r2.get("corrected", np.inf) < r2.get("fixed", -np.inf):
        warnings.append(f"{MARK_LABELS[ch]}: corrected parameters don't make size less density-dependent "
                        f"(R² {r2.get('fixed', np.nan):.2f} fixed vs {r2.get('corrected', np.nan):.2f} corrected)")
    if c.get("gt_frac_domains_overlapping", 0) > 0.5:
        warnings.append(f"{MARK_LABELS[ch]}: {c['gt_frac_domains_overlapping']:.0%} of ground-truth domains overlap "
                        "their nearest neighbour, so domain size is ill-defined (see calibrate.sweep_domain_scale)")
run_time = min(p.stat().st_mtime for p in cal_dir.glob("*_calibration.json"))
newer = sorted(p.name for p in (REPO_ROOT / "hmt_v3").rglob("*.py") if p.stat().st_mtime > run_time)
if newer:
    warnings.append(f"hmt_v3 changed after this run ({', '.join(newer)}); re-run the drivers so the figures "
                    "reflect the current code")

print(f"run: {RUN_DIR} (calibrated {time.strftime('%Y-%m-%d', time.localtime(run_time))}); nucleus {cell_id}, "
      f"{', '.join(f'{len(cell[ch]):,} {MARK_LABELS[ch]}' for ch in CHANNELS)} localizations")
for w in warnings:
    print("WARNING:", w)
pd.DataFrame(qc_rows).round(2)

## Figure 1. Data and extracted parameters

**a** Both marks in the nucleus, with the nucleus mask outlined and the zoom window of **b** boxed.
**b** Zoom. **c** The mask divided into 100 radial contours, core to periphery.

These are the statistics the simulation is built from (`hmt.simulate.generate_nucleus`):

**d** Radial density profile, which sets where domains are placed.
**e** Within-domain radial distribution (RDF): the neighbour density above background around each
localization. Out to the domain scale (dotted line) it sets each domain's lateral profile, and its integral
sets the localizations per domain.
**f** Axial distribution (ADF) of |Δz| between neighbours, which sets each domain's axial profile.
**g** Pair correlation g(r) against a uniform (CSR) pattern in the same mask. The noise fraction and the
clustering of domain centres (Thomas process) are fitted to it.

The RDF and ADF use `N_PARAMS` random localizations per mark, with the RDF rescaled to full density; g(r)
uses `N_PCF`. The values written on the panels are the ones the drivers' simulation used. They're listed in
the table after the figure.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)


def subsample(df, n):
    return df if len(df) <= n else df.sample(n, random_state=int(rng.integers(2**31)))


def simulation_inputs(locs):
    # The statistics generate_nucleus reads from one mark: radial profile, within-domain RDF / ADF, g(r)
    profile = hmt.measure.extract_radial_density_profile(locs, cell["bands"], cell["x_min"], cell["y_min"],
                                                         bin_size=cell["bin_nm"])
    s = subsample(locs, N_PARAMS)
    rdf, adf = hmt.measure.extract_empirical_parameters(s, **RDF)
    pcf = hmt.measure.measure_pair_correlation(subsample(locs, N_PCF), cell["mask"], cell["x_min"], cell["y_min"],
                                               px_size=cell["bin_nm"], **PCF)
    return {"profile": profile / profile.max(),
            "rdf": rdf * len(locs) / len(s) * 1e6,  # excess neighbour density at full labelling, per um^2
            "adf": adf / adf.sum(), "pcf": pcf}


t0 = time.time()
real_inputs = {ch: simulation_inputs(cell[ch]) for ch in CHANNELS}
np.random.seed(RANDOM_SEED)  # add_noise_locs draws from NumPy's global random state
csr = hmt.simulate.add_noise_locs(N_PCF, cell["bands"], np.zeros(1), cell["x_min"], cell["y_min"],
                                  px_size=cell["bin_nm"])
csr_pcf = hmt.measure.measure_pair_correlation(csr, cell["mask"], cell["x_min"], cell["y_min"],
                                               px_size=cell["bin_nm"], **PCF)
print(f"Figure 1 statistics: {time.time() - t0:.0f} s")

rdf_r = np.arange(RDF["step"], RDF["sdis"] + RDF["step"], RDF["step"]) - RDF["step"] / 2  # ring centres
adf_dz = np.arange(0, RDF["sdis"], RDF["step"]) + RDF["step"] / 2
band_pct = np.arange(len(real_inputs[CHANNELS[0]]["profile"])) + 0.5  # band b spans b..b+1 % core -> periphery
zoom = window(np.mean(np.reshape(calib[CHANNELS[0]]["crop_bounds_nm"], (2, 2)), axis=1))  # calibration crop centre
ext = nucleus_extent(cell)

In [ ]:
fig = plt.figure(figsize=(FIG_WIDTH_IN, 5.3), layout="constrained")
top, bottom = fig.subfigures(2, 1, height_ratios=[1.35, 1])
ax_img, ax_zoom, ax_bands = top.subplots(1, 3)
ax_prof, ax_rdf, ax_adf, ax_g = bottom.subplots(1, 4)

show_render(ax_img, {ch: cell[ch] for ch in CHANNELS}, ext, px_nm=25.0, legend=True)
mask_outline(ax_img, cell)
ax_img.add_patch(Rectangle((um(zoom[0]), um(zoom[2])), um(WINDOW_NM), um(WINDOW_NM), fill=False, ec="white", lw=0.8))
image_panel(ax_img, "Nucleus", 5)

show_render(ax_zoom, {ch: in_window(cell[ch], zoom) for ch in CHANNELS}, zoom, px_nm=10.0, blur_px=1.2)
image_panel(ax_zoom, f"Zoom ({um(WINDOW_NM):g} µm)", 0.5)

bands = np.ma.masked_less(cell["bands"].T.astype(float), 0)
b = cell["bin_nm"]
ax_bands.imshow(bands, origin="lower", cmap=SEQ_CMAP.reversed(), interpolation="nearest",
                extent=(um(cell["x_min"]), um(cell["x_min"] + bands.shape[1] * b),
                        um(cell["y_min"]), um(cell["y_min"] + bands.shape[0] * b)))
xc = um(cell["x_min"] + (np.arange(bands.shape[1]) + 0.5) * b)
yc = um(cell["y_min"] + (np.arange(bands.shape[0]) + 0.5) * b)
ax_bands.contour(xc, yc, bands.filled(-1), levels=[24.5, 49.5, 74.5], colors="white", linewidths=0.5)
set_window(ax_bands, ext)
image_panel(ax_bands, "Radial contours (dark = core)", 5, bar_color=INK)

for ch in CHANNELS:
    c, d = MARK_COLORS[ch], real_inputs[ch]
    ax_prof.plot(band_pct, d["profile"], color=c, label=MARK_LABELS[ch])
    ax_rdf.plot(rdf_r, d["rdf"], color=c, label=MARK_LABELS[ch])
    ax_rdf.axvline(info[ch]["r_domain_nm"], color=c, ls=":", lw=1)
    ax_adf.plot(adf_dz, d["adf"], color=c, label=MARK_LABELS[ch])
    ax_g.plot(d["pcf"]["r"] - PCF["dr"] / 2, d["pcf"]["g"], color=c, label=MARK_LABELS[ch])
ax_g.plot(csr_pcf["r"] - PCF["dr"] / 2, csr_pcf["g"], color=MUTED, ls="--", lw=1, label="CSR")
ax_g.axhline(1, color=AXIS, lw=0.8)

ax_rdf.text(0.97, 0.95, "\n".join(["locs / domain"] + [f"{MARK_LABELS[ch]} {info[ch]['n_locs_per_domain']:.0f}"
                                                        for ch in CHANNELS]),
            transform=ax_rdf.transAxes, ha="right", va="top", color=INK_2, fontsize=6)
ax_g.text(0.97, 0.62, "\n".join(["noise fraction"] + [f"{MARK_LABELS[ch]} {info[ch]['f_noise']:.0%}" for ch in CHANNELS]),
          transform=ax_g.transAxes, ha="right", va="top", color=INK_2, fontsize=6)
style(ax_prof, "Radius (%), core → periphery", "Density (norm.)", "Radial profile")
style(ax_rdf, "r (nm)", "Excess neighbours (µm⁻²)", "Within-domain RDF")
style(ax_adf, "|Δz| (nm)", "Fraction of pairs", "Axial distribution (ADF)")
style(ax_g, "r (nm)", "g(r)", "Pair correlation")
ax_prof.legend(frameon=False, loc="lower center")
ax_g.legend(frameon=False, loc="upper right")

letter_panels([ax_img, ax_zoom, ax_bands, ax_prof, ax_rdf, ax_adf, ax_g])
save(fig, "figure1_data_and_parameters")
plt.show()

params = pd.DataFrame({MARK_LABELS[ch]: {
    "domain scale (nm)": info[ch]["r_domain_nm"],
    "localizations per domain": info[ch]["n_locs_per_domain"],
    "domains": info[ch]["n_domains"],
    "noise fraction": info[ch]["f_noise"],
    "domain-centre clustering σ (nm)": info[ch].get("thomas_sigma_nm"),
    "domain-centre groups (per µm²)": (info[ch].get("thomas_kappa_p") or np.nan) * 1e6,
    "seed placement": info[ch].get("placement"),
} for ch in CHANNELS})
params.to_csv(FIG_DIR / "figure1_parameters.csv")
params

## Figure 2. Building the simulation

`hmt.simulate.generate_nucleus` rebuilds the nucleus from the parameters in Fig. 1, with every
localization's true domain known. Panels **a–c** show the steps for H3K27me3 in the zoom window of Fig. 1b:

**a** Domain centres are placed along the radial profile and grouped by a Thomas cluster process fitted to g(r).
**b** Each domain gets its localizations, drawn from the within-domain RDF (lateral) and ADF (axial).
**c** Uniform background is added at the fitted noise fraction. This is the simulated nucleus.
**d–e** The real and simulated nucleus, rendered the same way.
**f–i** Checks that the simulation reproduces the real statistics: real (solid) and simulated (dashed) radial
profile, RDF, ADF and g(r), all recomputed here with the same estimators.

In [ ]:
t0 = time.time()
sim_inputs = {ch: simulation_inputs(sims[ch]) for ch in CHANNELS}
print(f"Figure 2 statistics: {time.time() - t0:.0f} s")

steps_mark = CHANNELS[0]
sim_w = in_window(sims[steps_mark], zoom)
domain_locs = sim_w[sim_w["cluster_label"] >= 0]
noise_locs = sim_w[sim_w["cluster_label"] < 0]
seeds = sims[steps_mark][sims[steps_mark]["cluster_label"] >= 0].groupby("cluster_label")[["x [nm]", "y [nm]"]].mean()
seeds_w = in_window(seeds, zoom)

In [ ]:
fig = plt.figure(figsize=(FIG_WIDTH_IN, 7.0), layout="constrained")
row1, row2, row3 = fig.subfigures(3, 1, height_ratios=[1, 1.15, 0.9])
ax_seed, ax_dom, ax_full = row1.subplots(1, 3)
ax_real, ax_sim, ax_prof = row2.subplots(1, 3)
ax_rdf, ax_adf, ax_g = row3.subplots(1, 3)

gi = info[steps_mark]
for ax in (ax_seed, ax_dom, ax_full):
    set_window(ax, zoom)
scatter_locs(ax_full, noise_locs, BACKGROUND_COLOR, s=0.6, label="background")
for ax in (ax_dom, ax_full):
    scatter_locs(ax, domain_locs, MARK_COLORS[steps_mark], s=0.6, label="domain localizations")
for ax, s in ((ax_seed, 6), (ax_dom, 3)):
    scatter_locs(ax, seeds_w, INK, s=s, label="domain centres")
sigma = gi.get("thomas_sigma_nm")
image_panel(ax_seed, "Domain centres" + (f" (Thomas, σ = {sigma:.0f} nm)" if sigma else ""), 0.5, bar_color=INK)
image_panel(ax_dom, f"+ localizations ({gi['n_locs_per_domain']:.0f} / domain)", 0.5, bar_color=INK)
image_panel(ax_full, f"+ background ({gi['f_noise']:.0%})", 0.5, bar_color=INK)
ax_full.legend(loc="upper left", frameon=True, framealpha=0.85, edgecolor="none", markerscale=4, fontsize=6)

show_render(ax_real, {ch: cell[ch] for ch in CHANNELS}, ext, px_nm=25.0, legend=True)
show_render(ax_sim, {ch: sims[ch] for ch in CHANNELS}, ext, px_nm=25.0)
for ax, title in ((ax_real, "Real"), (ax_sim, "Simulated")):
    mask_outline(ax, cell)
    image_panel(ax, title, 5)

for ch in CHANNELS:
    c = MARK_COLORS[ch]
    for inputs, ls in ((real_inputs, "-"), (sim_inputs, "--")):
        d = inputs[ch]
        ax_prof.plot(band_pct, d["profile"], color=c, ls=ls)
        ax_rdf.plot(rdf_r, d["rdf"], color=c, ls=ls)
        ax_adf.plot(adf_dz, d["adf"], color=c, ls=ls)
        ax_g.plot(d["pcf"]["r"] - PCF["dr"] / 2, d["pcf"]["g"], color=c, ls=ls)
ax_g.axhline(1, color=AXIS, lw=0.8)
style(ax_prof, "Radius (%), core → periphery", "Density (norm.)", "Radial profile")
style(ax_rdf, "r (nm)", "Excess neighbours (µm⁻²)", "Within-domain RDF")
style(ax_adf, "|Δz| (nm)", "Fraction of pairs", "Axial distribution (ADF)")
style(ax_g, "r (nm)", "g(r)", "Pair correlation")
key = [Line2D([], [], color=MARK_COLORS[ch], label=MARK_LABELS[ch]) for ch in CHANNELS] + \
      [Line2D([], [], color=INK_2, ls="-", label="real"), Line2D([], [], color=INK_2, ls="--", label="simulated")]
ax_prof.legend(handles=key, frameon=False, loc="lower center", ncol=2, fontsize=6)

letter_panels([ax_seed, ax_dom, ax_full, ax_real, ax_sim, ax_prof, ax_rdf, ax_adf, ax_g])
save(fig, "figure2_simulation")
plt.show()

## Figure 3. Calibration

DBSCAN finds domains of the right size only if eps and min_samples suit the labelling density. The
calibration learns how they should change with density, on the simulation, where the true domains are known
(`hmt.calibrate.optimize_dbscan_size_matching`).

**a** The simulated nucleus, with each mark's calibration crop: a square whose density matches the nucleus
mean.
**b–d** The H3K27me3 crop thinned to lower labelling densities. The domains stay the same; only the number
of localizations per domain changes. True-domain localizations are in colour, background in grey.
**e–f** The search at two densities. Each cell is DBSCAN's size error for one (eps, min_samples): the
|mean cluster p75 diameter − true diameter|. The ring marks the pair the calibration chose (after refining
eps). Grey cells found no clusters.
**g–h** The chosen eps and min_samples at every density, with the fitted eps = a·densityᵇ and
min_samples = m·density + c. Open markers are densities whose best size error exceeded
`SIZE_COST_THRESHOLD_NM` and were left out of the fit.
**i–j** Measured domain size against density on the simulated crop, with parameters fixed at their
full-density values (**i**) and corrected for density (**j**). The dashed lines are the true diameters. With
a working calibration, corrected sizes stay flat and close to them.

Panels e–f are recomputed here with `hmt.calibrate.domain_size_cost` on the calibration's coarse eps grid
(about 20 s per map). The other panels plot the drivers' saved calibration.

In [ ]:
crop_rng = np.random.default_rng(RANDOM_SEED)
lc = calib[LANDSCAPE_MARK]
cx0, cx1, cy0, cy1 = lc["crop_bounds_nm"]
s = sims[LANDSCAPE_MARK]
crop = s[s["x [nm]"].between(cx0, cx1, inclusive="left") & s["y [nm]"].between(cy0, cy1, inclusive="left")]
crop_area = lc["calibration_area_nm2"]
thinned = {f: crop if f >= 1 else crop.sample(frac=f, random_state=int(crop_rng.integers(2**31)))
           for f in sorted(set(CROP_FRACTIONS) | set(LANDSCAPE_FRACTIONS), reverse=True)}

t0 = time.time()
landscapes = {}
for f in LANDSCAPE_FRACTIONS:
    coords = thinned[f][["x [nm]", "y [nm]", "z [nm]"]].to_numpy()
    cost = np.array([[hmt.calibrate.domain_size_cost(eps, coords, lc["gt_target"], ms, percentile=C.SIZE_PERCENTILE,
                                                     use_z=lc["use_z"])
                      for eps in LANDSCAPE_EPS_NM] for ms in lc["min_samples_range"]])
    # the calibration's own best pair at this density (its saved fractions are matched by density)
    density = len(coords) / crop_area
    k = int(np.argmin(np.abs(np.array(lc["all_densities"]) - density)))
    landscapes[f] = {"cost": cost, "best": (lc["all_epsilons"][k], lc["all_min_samples"][k])}
print(f"size-error maps: {time.time() - t0:.0f} s")

In [ ]:
fig = plt.figure(figsize=(FIG_WIDTH_IN, 6.6), layout="constrained")
row1, row2, row3 = fig.subfigures(3, 1, height_ratios=[1, 1.05, 1])
ax_nuc, *ax_crops = row1.subplots(1, 1 + len(CROP_FRACTIONS))
ax_l = list(row2.subplots(1, len(LANDSCAPE_FRACTIONS) + 2))
ax_land, ax_eps, ax_ms = ax_l[:len(LANDSCAPE_FRACTIONS)], ax_l[-2], ax_l[-1]
ax_fixed, ax_corr = row3.subplots(1, 2, sharey=True)

# a: simulated nucleus with both calibration crops
show_render(ax_nuc, {ch: sims[ch] for ch in CHANNELS}, ext, px_nm=25.0)
mask_outline(ax_nuc, cell, color=MUTED)
for ch in CHANNELS:
    x0, x1, y0, y1 = calib[ch]["crop_bounds_nm"]
    ax_nuc.add_patch(Rectangle((um(x0), um(y0)), um(x1 - x0), um(y1 - y0), fill=False, ec=MARK_COLORS[ch], lw=1.0))
image_panel(ax_nuc, "Calibration crops", 5)

# b-d: the crop thinned
for ax, f in zip(ax_crops, CROP_FRACTIONS):
    d = thinned[f]
    dom = d["cluster_label"] >= 0
    scatter_locs(ax, d[~dom], BACKGROUND_COLOR, s=0.25)
    scatter_locs(ax, d[dom], MARK_COLORS[LANDSCAPE_MARK], s=0.25)
    set_window(ax, lc["crop_bounds_nm"])
    image_panel(ax, f"{f:.0%} ({len(d) / crop_area * 1e6:.0f} locs µm⁻²)", 1, bar_color=INK)

# e-f: size-error maps
ms_vals = np.asarray(lc["min_samples_range"])
deps, dms = LANDSCAPE_EPS_NM[1] - LANDSCAPE_EPS_NM[0], ms_vals[1] - ms_vals[0]
for ax, f in zip(ax_land, LANDSCAPE_FRACTIONS):
    L = landscapes[f]
    ax.set_facecolor(GRID)  # cells masked out below (no clusters found) show this
    im = ax.imshow(np.ma.masked_greater_equal(L["cost"], 1e5), origin="lower", aspect="auto", cmap=SEQ_CMAP,
                   vmin=0, vmax=lc["gt_target"], interpolation="nearest",
                   extent=(LANDSCAPE_EPS_NM[0] - deps / 2, LANDSCAPE_EPS_NM[-1] + deps / 2,
                           ms_vals[0] - dms / 2, ms_vals[-1] + dms / 2))
    ax.plot(*L["best"], "o", ms=7, mfc="none", mec=INK, mew=1.3)
    style(ax, "eps (nm)", "min_samples" if ax is ax_land[0] else "", f"{f:.0%} density")
    ax.grid(False)
row2.colorbar(im, ax=ax_land, label="Size error (nm)", extend="max", shrink=0.85, aspect=25)

# g-h: chosen parameters and fits
for ch in CHANNELS:
    c, cc = MARK_COLORS[ch], calib[ch]
    dens = np.array(cc["all_densities"]) * 1e6
    kept = np.array(cc["all_size_errors"]) < C.SIZE_COST_THRESHOLD_NM
    for ax, vals in ((ax_eps, cc["all_epsilons"]), (ax_ms, cc["all_min_samples"])):
        vals = np.asarray(vals, float)
        ax.scatter(dens[kept], vals[kept], s=14, color=c, zorder=3, label=MARK_LABELS[ch])
        ax.scatter(dens[~kept], vals[~kept], s=14, facecolors="none", edgecolors=c, zorder=3)
    dd = np.linspace(dens.min(), dens.max(), 100)
    ax_eps.plot(dd, cc["eps_coef"] * (dd / 1e6) ** cc["eps_exp"], color=c, lw=1)
    ax_ms.plot(dd, cc["min_samples_slope"] * dd / 1e6 + cc["min_samples_intercept"], color=c, lw=1)
style(ax_eps, "Density (locs µm⁻²)", "eps (nm)", "Chosen eps")
style(ax_ms, "Density (locs µm⁻²)", "min_samples", "Chosen min_samples")
ax_eps.legend(frameon=False, loc="upper right")

# i-j: size vs density, fixed vs corrected
for ax, method, title in ((ax_fixed, "fixed", "Fixed parameters"), (ax_corr, "corrected", "Density-corrected")):
    notes = []
    st = METHOD_STYLE[method]
    for ch, table in size_tables.items():
        c = MARK_COLORS[ch]
        t = table[table["method"] == method].dropna(subset=["mean_diameter_nm"])
        ax.plot(t["density_per_um2"], t["mean_diameter_nm"], marker="o", ls=st["ls"], color=c, ms=3.5, lw=1,
                mfc=c if st["filled"] else "none", label=MARK_LABELS[ch])
        ax.axhline(calib[ch]["gt_target"], color=c, ls=":", lw=0.9)
        r2 = linregress(t["density_per_um2"], t["mean_diameter_nm"]).rvalue ** 2 if len(t) > 2 else np.nan
        notes.append(f"{MARK_LABELS[ch]}: R² = {r2:.2f}")
    ax.text(0.98, 0.04, "\n".join(notes), transform=ax.transAxes, ha="right", va="bottom", color=INK_2, fontsize=6)
    style(ax, "Density (locs µm⁻²)", "Mean domain diameter, p75 (nm)" if ax is ax_fixed else "", title)
ax_fixed.legend(frameon=False, loc="upper right")

letter_panels([ax_nuc, *ax_crops, *ax_land, ax_eps, ax_ms, ax_fixed, ax_corr])
save(fig, "figure3_calibration")
plt.show()

## Figure 4. Clustering the nucleus

The calibration from Fig. 3 applied to the real nucleus of Fig. 1 (`hmt.cluster.cluster_dbscan_density_corrected`,
with the drivers' `MIN_EPS_NM` / `MIN_MIN_SAMPLES` floors).

**a–b** The fitted eps(density) and min_samples(density) for each mark over the calibrated density range,
and the values read off at this nucleus's density (diamond).
**c** The nucleus clustered with those parameters: clustered localizations in colour, unclustered in grey.
**d–e** Each mark in the zoom window of Fig. 1b, with every nanodomain's outline (convex hull of its
localizations in the window).
**f–h** The fixed-vs-corrected check of Fig. 3i–j, repeated on the real nucleus. The nucleus is thinned to
`REAL_FRACTIONS` of its localizations and clustered with parameters fixed at full density (dashed, open) or
corrected for the thinned density (solid, filled). Thinning removes localizations, not domains, so a working
calibration keeps the measured size level (**h**) where fixed parameters let it drift (**g**). Panel **f**
counts the nanodomains each run finds, relative to full density. Domains and sizes use nanodomains with at
least `MIN_DOMAIN_LOCS` localizations.

In [ ]:
cluster_kw = dict(use_z=calib[CHANNELS[0]]["use_z"], min_eps=C.MIN_EPS_NM, min_min_samples=C.MIN_MIN_SAMPLES)
area = cell["area_nm2"]


def domain_summary(clustered):
    nd = hmt.cluster.calculate_nanodomain_characteristics(clustered, percentile=C.SIZE_PERCENTILE)
    nd = nd[nd["n_locs"] >= MIN_DOMAIN_LOCS]
    return (nd["geometric_diameter_2D"].mean() if len(nd) else np.nan), len(nd)


t0 = time.time()
thin_rng = np.random.default_rng(RANDOM_SEED)
clustered, used, rows = {}, {}, []
for ch in CHANNELS:
    clustered[ch], used[ch] = hmt.cluster.cluster_dbscan_density_corrected(cell[ch], calib[ch], area, **cluster_kw)
    for f in REAL_FRACTIONS:
        locs = cell[ch] if f >= 1 else cell[ch].sample(frac=f, random_state=int(thin_rng.integers(2**31)))
        runs = {"fixed": hmt.cluster.cluster_dbscan(locs, eps=used[ch]["eps"], min_samples=used[ch]["min_samples"],
                                                    use_z=cluster_kw["use_z"]),
                "corrected": hmt.cluster.cluster_dbscan_density_corrected(locs, calib[ch], area, **cluster_kw)[0]}
        for method, result in runs.items():
            size, n = domain_summary(result)
            rows.append({"mark": ch, "fraction": f, "density_per_um2": len(locs) / area * 1e6, "method": method,
                         "mean_diameter_nm": size, "n_domains": n})
thinning = pd.DataFrame(rows)
thinning["n_domains_rel"] = thinning["n_domains"] / thinning.groupby(["mark", "method"])["n_domains"].transform("first")
thinning.to_csv(FIG_DIR / "figure4_thinning.csv", index=False)
print(f"clustering: {time.time() - t0:.0f} s")
pd.DataFrame({MARK_LABELS[ch]: {"density (locs µm⁻²)": used[ch]["density"] * 1e6, "eps (nm)": used[ch]["eps"],
                                "min_samples": used[ch]["min_samples"],
                                "nanodomains": domain_summary(clustered[ch])[1]} for ch in CHANNELS}).round(1)

In [ ]:
fig = plt.figure(figsize=(FIG_WIDTH_IN, 7.0), layout="constrained")
row1, row2, row3 = fig.subfigures(3, 1, height_ratios=[1.1, 1.05, 0.9])
ax_eps, ax_ms, ax_nuc = row1.subplots(1, 3, width_ratios=[1, 1, 1.15])
ax_z = list(row2.subplots(1, len(CHANNELS) + 1))
ax_zoom, ax_count = ax_z[:len(CHANNELS)], ax_z[-1]
ax_fixed, ax_corr = row3.subplots(1, 2, sharey=True)

# a-b: the calibration curves and this nucleus's operating point
for ch in CHANNELS:
    c, cc, d = MARK_COLORS[ch], calib[ch], used[ch]["density"] * 1e6
    dd = np.linspace(min(cc["all_densities"]) * 1e6, max(max(cc["all_densities"]) * 1e6, d), 100)
    ax_eps.plot(dd, cc["eps_coef"] * (dd / 1e6) ** cc["eps_exp"], color=c, lw=1.2, label=MARK_LABELS[ch])
    ax_ms.plot(dd, cc["min_samples_slope"] * dd / 1e6 + cc["min_samples_intercept"], color=c, lw=1.2,
               label=MARK_LABELS[ch])
    for ax, val, fmt in ((ax_eps, used[ch]["eps"], "{:.0f} nm"), (ax_ms, used[ch]["min_samples"], "{:.0f}")):
        ax.plot(d, val, "D", ms=5, color=c, mec=INK, mew=0.7, zorder=4)
        ax.annotate(fmt.format(val), (d, val), xytext=(-6, 4), textcoords="offset points", ha="right",
                    va="bottom", color=INK_2, fontsize=6)
style(ax_eps, "Density (locs µm⁻²)", "eps (nm)", "eps at this density")
style(ax_ms, "Density (locs µm⁻²)", "min_samples", "min_samples at this density")
ax_eps.legend(frameon=False, loc="upper right")

# c: clustered nucleus
for ch in CHANNELS:
    lab = clustered[ch]["cluster_label"] >= 0
    scatter_locs(ax_nuc, clustered[ch][~lab], BACKGROUND_COLOR, s=0.05, zorder=1)
for ch in CHANNELS:
    lab = clustered[ch]["cluster_label"] >= 0
    scatter_locs(ax_nuc, clustered[ch][lab], MARK_COLORS[ch], s=0.05, zorder=2)
mask_outline(ax_nuc, cell, color=MUTED)
ax_nuc.add_patch(Rectangle((um(zoom[0]), um(zoom[2])), um(WINDOW_NM), um(WINDOW_NM), fill=False, ec=INK, lw=0.8))
set_window(ax_nuc, ext)
image_panel(ax_nuc, "Clustered nucleus", 5, bar_color=INK)

# d-e: zoom per mark with nanodomain outlines
for ax, ch in zip(ax_zoom, CHANNELS):
    w = in_window(clustered[ch], zoom)
    lab = w["cluster_label"] >= 0
    scatter_locs(ax, w[~lab], BACKGROUND_COLOR, s=0.8)
    scatter_locs(ax, w[lab], MARK_COLORS[ch], s=0.8)
    for _, g in w[lab].groupby("cluster_label"):
        xy = um(g[["x [nm]", "y [nm]"]].to_numpy())
        if len(xy) >= 3:
            hull = ConvexHull(xy)
            ax.fill(*xy[hull.vertices].T, fc="none", ec=INK, lw=0.5)
    set_window(ax, zoom)
    image_panel(ax, f"{MARK_LABELS[ch]} nanodomains", 0.5, bar_color=INK)

# f-h: thinning the real nucleus
for ch in CHANNELS:
    c = MARK_COLORS[ch]
    for method, st in METHOD_STYLE.items():
        t = thinning[(thinning["mark"] == ch) & (thinning["method"] == method)]
        kw = dict(marker="o", ls=st["ls"], color=c, ms=3.5, lw=1, mfc=c if st["filled"] else "none")
        ax_count.plot(t["fraction"] * 100, t["n_domains_rel"], **kw)
        (ax_fixed if method == "fixed" else ax_corr).plot(t["fraction"] * 100, t["mean_diameter_nm"], **kw,
                                                         label=MARK_LABELS[ch])
ax_count.axhline(1, color=AXIS, lw=0.8)
style(ax_count, "Localizations kept (%)", "Nanodomains (rel. to 100%)", "Nanodomains found")
method_key = [Line2D([], [], color=INK_2, ls=st["ls"], marker="o", ms=3.5, mfc=INK_2 if st["filled"] else "none",
                     label=m) for m, st in METHOD_STYLE.items()]
ax_count.legend(handles=method_key, frameon=False, loc="lower right")
for ax, method, title in ((ax_fixed, "fixed", "Fixed parameters"), (ax_corr, "corrected", "Density-corrected")):
    notes = []
    for ch in CHANNELS:
        t = thinning[(thinning["mark"] == ch) & (thinning["method"] == method)].dropna(subset=["mean_diameter_nm"])
        r2 = linregress(t["density_per_um2"], t["mean_diameter_nm"]).rvalue ** 2 if len(t) > 2 else np.nan
        notes.append(f"{MARK_LABELS[ch]}: R² = {r2:.2f}")
    ax.text(0.98, 0.04, "\n".join(notes), transform=ax.transAxes, ha="right", va="bottom", color=INK_2, fontsize=6)
    style(ax, "Localizations kept (%)", "Mean domain diameter, p75 (nm)" if ax is ax_fixed else "", title)
ax_fixed.legend(frameon=False, loc="upper left")

letter_panels([ax_eps, ax_ms, ax_nuc, *ax_zoom, ax_count, ax_fixed, ax_corr])
save(fig, "figure4_clustering")
plt.show()
print(f"figures -> {FIG_DIR}")